# OpenAire raw EDA
 
## Setup

In [1]:
import sys
sys.path.insert(0, '/home/lu72hip/DIGICHer/dh_pipeline/src')
import pandas as pd
from pathlib import Path
from lib.database.duck.create_connection import create_duck_connection
from utils.config.config_loader import get_query_config

config = get_query_config()['openaire_dump']
# DB = Path(config['path_duck'])
DB = Path(config["path_duck_staging"])

con = create_duck_connection(str(DB))
con.execute("SET memory_limit='64GB'")
con.execute("SET threads=16")
print(f'Connected to {DB}')

pd.set_option("display.max_rows", 21) # Show more rows  # or None for unlimited
pd.set_option("display.max_columns", None) # Show more columns  # None = show all
# pd.set_option("display.max_colwidth", None) # Don't truncate column content  # None = full content
# pd.set_option("display.width", None) # Wider display  # Auto-detect terminal width

/home/lu72hip/DIGICHer/dh_pipeline/config/config_queries.json {'arxiv': {'checkpoint': 'submittedDateTo', 'queries': [{'query': '(all:cultural+OR+all:heritage+OR+all:digital+OR+all:humanities)', 'download_attachments': True, 'checkpoint_start': '1990-01-01-00-00', 'checkpoint_range': '6'}, {'query': 'all:computing+AND+(all:humanities+OR+all:heritage)', 'download_attachments': True, 'checkpoint_start': '1990-01-01-00-00', 'checkpoint_range': '6'}]}, 'cordis': {'checkpoint': 'startDate', 'queries': [{'query': 'cultural OR heritage OR digital OR humanities', 'download_attachments': True, 'checkpoint_start': '1985-01-01', 'checkpoint_range': '1'}, {'query': "contenttype='project' AND '*'", 'download_attachments': False, 'checkpoint_start': '1985-01-01', 'checkpoint_range': '1', 'path_duck': '/vast/lu72hip/data/duckdb/sources/cordis_raw.duckdb'}]}, 'coreac': {'checkpoint': 'publishedDate', 'queries': [{'query': '((computing AND cultural) OR (computing AND heritage))', 'download_attachments'

## General Stats

In [2]:
### Get counts of all tables

q = """ 
SELECT
    table_name,
    estimated_size as cnt
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY estimated_size DESC
"""
con.execute(q).df()

,table_name,cnt
0,relation,562206510
1,work,205841448
2,project,3656773
3,organization,448161


In [4]:
### Hash duplication detection
# Result: No collisions :)

q = """
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT id) as unique_ids,
    COUNT(DISTINCT hash(id)) as unique_hashes,
    total_rows - unique_hashes as collisions
FROM work
"""
display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_ids,unique_hashes,collisions
0,205841448,205841448,205841448,0


## Entity Organization

### Geolocation

* 440k orgs in total
* 120k ROR orgs 
--> For Beta only get geolocations for ROR

* GRID 
* PIC 70342
* ISNI 60138
* Wikidata 57673 -> ROR overlap
* mag_id 27080
* FundRef 22750
* OrgRef 16448
* RNSR 9157
* OrgReg 6639
--> After BETA calculate overlap of other schemas with ROR, see if u can get geolocations from scheme source
--> For remaining orgs use Geocoding services with name + country

### Sanitization Organization

* legalName: Asc and Desc -> Good, but needs legalName sanitization
* BOM needs to be cleaned in text, eg value.lstrip('\ufeff\u200b\u200c\u200d\ufffe')

In [3]:
###  Sample Rows

q = """ 
SELECT * FROM organization o
LIMIT 3;
"""
display(con.execute(q).df())

,id,openaireId,legalName,legalShortName,websiteUrl,alternativeNames,countryCode,rorId,wikiId
0,1934433190169772723,openorgs____::05a9c3805e5e7d8933851bf80982e400,BONFIGLIOLI RIDUTTORI SPA,BRI,http://www.bonfiglioli.com,[BRI],IT,None,None
1,16526272328665692646,openorgs____::14661c2b103c69736d711c1ca49c0602,Mid Michigan Community College,MMCC,https://www.midmich.edu/,"[Mid Michigan Community College, MMCC]",US,https://ror.org/03f3r3r28,Q6840980
2,40960094163156804,openorgs____::1c063fea076f9fd2dbfd87faf906c647,International Fund for Animal Welfare,IFAW,http://www.ifaw.org/united-states,"[International Fund for Animal Welfare, IFAW]",US,https://ror.org/043c66817,Q1666594


In [6]:
### How often each Oragnization ID scheme is used across all orgs

q = """ 
SELECT 
    pid.scheme,
    COUNT(DISTINCT organization.id) as orgs_with_scheme,
    ROUND(100.0 * COUNT(DISTINCT organization.id) / (SELECT COUNT(*) FROM organization), 2) as pct_coverage
FROM organization,
UNNEST(pids) as t(pid)
GROUP BY pid.scheme
ORDER BY orgs_with_scheme DESC
"""
display(con.execute(q).df())

,scheme,orgs_with_scheme,pct_coverage
0,ROR,123542,27.57
1,GRID,108191,24.14
2,PIC,69287,15.46
3,ISNI,59194,13.21
4,Wikidata,56164,12.53
5,mag_id,26448,5.90
6,FundRef,17919,4.00
7,OrgRef,15148,3.38
8,OrgReg,6477,1.45
9,RNSR,6412,1.43


In [8]:
### Search FSU Jena

q = """ 
SELECT * FROM organization o
WHERE legalName like '%Schiller%'
AND legalShortName = 'FSU'
ORDER BY legalName DESC
LIMIT 2
"""
display(con.execute(q).df())

,legalShortName,legalName,websiteUrl,alternativeNames,country,id,pids
0,FSU,Friedrich Schiller University Jena,https://www.uni-jena.de/en/start.html,"[Friedrich-Schiller-Universität Jena, Friedric...","{'code': 'DE', 'label': 'Germany'}",openorgs____::f729b2897d3163c51a6fd5c9867c6dce,"[{'scheme': 'ISNI', 'value': '0000000119392794..."


In [10]:
### Sanitization: legalName
# * Needs to trim BOM

q = """ 
SELECT legalName FROM organization o
ORDER BY TRIM(REPLACE(legalName, chr(65279), '')) DESC -- 65279 = BOM
LIMIT 30;
"""
display(con.execute(q).df())

,legalName
0,中粮工科（西安）国际工程有限公司
1,上海交通大学出版社有限公司
2,《光通信研究》编辑部
3,《中国油脂》杂志社
4,Energy Dynamics (Norway)
...,...
25,​The Chinese Physiological Society
26,​Sevalanka Foundation
27,​Mercy College of Ohio
28,​Hail University


In [11]:
### Check legalName duplicates ###

q = """ 
SELECT legalName, country, COUNT(*) as cnt
FROM organization 
GROUP BY legalName, country
HAVING COUNT(*) > 1
ORDER BY cnt DESC"""
# display(con.execute(q).df())

### Same name 'Ministry of Health' for different countries ###

q = """ 
SELECT * FROM organization 
WHERE legalName = 'Ministry of Health'"""
# display(con.execute(q).df())

### See ROR specific duplicates ###

q = """ 
SELECT legalName, country, COUNT(*) as cnt
FROM organization 
WHERE list_contains(list_transform(pids, x -> x.scheme), 'ROR')
GROUP BY legalName, country
HAVING COUNT(*) > 1
ORDER BY cnt DESC
"""
# display(con.execute(q).df())

### Check Argosy University entries for duplicates in ROR
# * All 17 are subsidiaries/ departments of the same institution in different regions -> no dup

q = """
SELECT 
    id,
    legalName,
    legalShortName,
    websiteUrl,
    country,
    alternativeNames,
    pids
FROM organization 
WHERE legalName = 'Argosy University' 
  AND country.code = 'US'
  AND list_contains(list_transform(pids, x -> x.scheme), 'ROR')
ORDER BY id
"""
display(con.execute(q).df())

,id,legalName,legalShortName,websiteUrl,country,alternativeNames,pids
0,openorgs____::3dec1dbb744beb8db085730d02cb62bd,Argosy University,Argosy University,https://www.argosy.edu/,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'GRID', 'value': 'grid.441437.1'},..."
1,pending_org_::103e41938ad592409b0b322d0a8c0cb0,Argosy University,Argosy University,https://www.argosy.edu/locations/nashville,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'GRID', 'value': 'grid.472245.2'},..."
2,pending_org_::198d821956d4a7b3b6a8ed9e160320d7,Argosy University,Argosy University,https://www.argosy.edu/locations/chicago-downtown,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'OrgRef', 'value': '5899363'}, {'s..."
3,pending_org_::2c171a84133bdef229f2473a2a2a24a5,Argosy University,Argosy University,https://www.argosy.edu/locations/chicago-schau...,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'GRID', 'value': 'grid.441454.4'},..."
4,pending_org_::308a7ce1dbcadad2d5fcc3c285497662,Argosy University,Argosy University,http://argosy.edu/,"{'code': 'US', 'label': 'United States'}",<NA>,"[{'scheme': 'ISNI', 'value': '000000040389111X..."
5,pending_org_::3b604a19267967081c2c316842d2a4c3,Argosy University,Argosy University,https://www.argosy.edu/locations/hawaii,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'ISNI', 'value': '0000000403890715..."
6,pending_org_::5ab843685c0380cb4230b59bc8ad130d,Argosy University,Argosy University,https://www.argosy.edu/locations/dallas,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'Wikidata', 'value': 'Q30293684'},..."
7,pending_org_::6b3ca500ae14ce05558d33e83f1e4318,Argosy University,Argosy University,https://www.argosy.edu/locations/atlanta,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'ROR', 'value': 'https://ror.org/0..."
8,pending_org_::8258f6d5ca447f27b3ef3a9d93f3cc9f,Argosy University,Argosy University,https://www.argosy.edu/fullerton,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'GRID', 'value': 'grid.466460.6'},..."
9,pending_org_::835a06a4c5cf1dcb0765ab0385201088,Argosy University,Argosy University,https://www.argosy.edu/locations/tampa,"{'code': 'US', 'label': 'United States'}",[Argosy University],"[{'scheme': 'GRID', 'value': 'grid.472247.0'},..."


## Entity Project

In [10]:
### Sample Rows
# 226 publications with title 'unidentified'
# 16587 with title NULL and 'unidentified'

q = """ 
SELECT * FROM project
WHERE array_length(frameworkProgrammes) > 0
LIMIT 3
"""
display(con.execute(q).df())

### Look at specific columns
q = """ 
SELECT title, granted FROM project
WHERE granted is not NULL
LIMIT 15
"""
# display(con.execute(q).df())

,id,openaireId,grantId,title,acronym,websiteUrl,startDate,endDate,callIdentifier,keywords,openAccessMandateForPublications,openAccessMandateForDataset,subjects,fundings,frameworkProgrammes,summary,granted
0,16508330868004209722,arc_________::017f6ae69fd53f561c2a6fadca03357a,DP0451399,Understanding the structure of internal migrat...,None,http://purl.org/au-research/grants/arc/DP0451399,2004-01-01,2008-12-31,None,None,False,False,<NA>,"[{'shortName': 'ARC', 'name': 'Australian Rese...",[Discovery Projects],Understanding the structure of internal migrat...,"{'currency': 'AUD', 'totalCost': 0.0, 'fundedA..."
1,12618460875906795443,arc_________::01d40223496c2bb9652fcb3136c51209,DP200100475,Discovery Projects - Grant ID: DP200100475,None,http://purl.org/au-research/grants/arc/DP20010...,2020-01-01,2023-12-31,None,None,False,False,<NA>,"[{'shortName': 'ARC', 'name': 'Australian Rese...",[Discovery Projects],Caveospheres: A versatile peptide delivery sys...,"{'currency': 'AUD', 'totalCost': 0.0, 'fundedA..."
2,12189159823196739702,arc_________::01e131e2f08b7eea5ff63c060307a8dd,DP140101565,Discovery Projects - Grant ID: DP140101565,None,http://purl.org/au-research/grants/arc/DP14010...,2014-01-01,2018-12-31,None,None,False,False,<NA>,"[{'shortName': 'ARC', 'name': 'Australian Rese...",[Discovery Projects],Pushing The Boundaries Of Flow Chemistry &#821...,"{'currency': 'AUD', 'totalCost': 0.0, 'fundedA..."


In [8]:
### Project Table: Null/Empty Analysis ###

q = """
SELECT 
    'id' as column_name,
    COUNT(*) - COUNT(id) as null_count,
    ROUND(100.0 * (COUNT(*) - COUNT(id)) / COUNT(*), 2) as null_pct
FROM project
UNION ALL
SELECT 
    'code',
    COUNT(*) - COUNT(code),
    ROUND(100.0 * (COUNT(*) - COUNT(code)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'acronym',
    COUNT(*) - COUNT(acronym),
    ROUND(100.0 * (COUNT(*) - COUNT(acronym)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'title',
    COUNT(*) - COUNT(title),
    ROUND(100.0 * (COUNT(*) - COUNT(title)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'websiteUrl',
    COUNT(*) - COUNT(websiteUrl),
    ROUND(100.0 * (COUNT(*) - COUNT(websiteUrl)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'startDate',
    COUNT(*) - COUNT(startDate),
    ROUND(100.0 * (COUNT(*) - COUNT(startDate)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'endDate',
    COUNT(*) - COUNT(endDate),
    ROUND(100.0 * (COUNT(*) - COUNT(endDate)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'callIdentifier',
    COUNT(*) - COUNT(callIdentifier),
    ROUND(100.0 * (COUNT(*) - COUNT(callIdentifier)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'keywords',
    COUNT(*) - COUNT(keywords),
    ROUND(100.0 * (COUNT(*) - COUNT(keywords)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'openAccessMandateForPublications',
    COUNT(*) - COUNT(openAccessMandateForPublications),
    ROUND(100.0 * (COUNT(*) - COUNT(openAccessMandateForPublications)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'openAccessMandateForDataset',
    COUNT(*) - COUNT(openAccessMandateForDataset),
    ROUND(100.0 * (COUNT(*) - COUNT(openAccessMandateForDataset)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'subjects',
    COUNT(*) - COUNT(subjects),
    ROUND(100.0 * (COUNT(*) - COUNT(subjects)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'fundings',
    COUNT(*) - COUNT(fundings),
    ROUND(100.0 * (COUNT(*) - COUNT(fundings)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'summary',
    COUNT(*) - COUNT(summary),
    ROUND(100.0 * (COUNT(*) - COUNT(summary)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'granted',
    COUNT(*) - COUNT(granted),
    ROUND(100.0 * (COUNT(*) - COUNT(granted)) / COUNT(*), 2)
FROM project
UNION ALL
SELECT 
    'h2020Programmes',
    COUNT(*) - COUNT(h2020Programmes),
    ROUND(100.0 * (COUNT(*) - COUNT(h2020Programmes)) / COUNT(*), 2)
FROM project
ORDER BY null_count DESC
"""
display(con.execute(q).df())

In [14]:
### Filter EC as funder
q = """ 
SELECT 
    title, 
    unnest(fundings) as funding
FROM project
WHERE list_contains(
    list_transform(fundings, f -> f.shortName), 
    'EC'
)
LIMIT 3
"""
display(con.execute(q).df())

### Count Projects funded by EC
# Result: 115447

q = """
SELECT COUNT(DISTINCT id) AS ec_project_count
FROM project
WHERE list_contains(
    list_transform(fundings, f -> f.shortName),
    'EC'
);
"""
display(con.execute(q).df())

,title,funding
0,"Design, analysis and applications of novel inf...","{'shortName': 'EC', 'name': 'European Commissi..."
1,Functional and molecular characterization of e...,"{'shortName': 'EC', 'name': 'European Commissi..."
2,Production of Coatings for New Efficient and C...,"{'shortName': 'EC', 'name': 'European Commissi..."


,ec_project_count
0,115447


In [15]:
### Extract EU frameworkProgrammes
# Result| EC :: {FRAMEWORK} :: {CALL_TYPE}

q = """
SELECT DISTINCT 
    unnest(list_transform(fundings, f -> f.fundingStream.id)) as funding_stream_id
FROM project
WHERE list_contains(
    list_transform(fundings, f -> f.shortName), 
    'EC'
)
LIMIT 20
"""
display(con.execute(q).df())

,funding_stream_id
0,EC::HE::HORIZON-EIC\HORIZON-AG
1,EC::H2020::SME-1
2,EC::H2020::MSCA-IF-GF
3,EC::FP7::SP4::REGIONS
4,EC::FP7::SP1::SSH
5,EC::FP7::SP1::SPA
6,EC::HE::HORIZON-EIC-ACC-BF
7,EC::HE::EURATOM-RIA
8,EC::HE::HORIZON-JU-IA
9,EC::H2020::IA


In [ ]:
### How to extract FPs from EC Projects

q = """
CREATE TABLE projects_clean AS
SELECT 
    hash(id) as id,
    id as openaire_id,
    title,
    summary,
    callIdentifier as call_identifier,
    
    -- Extract EU framework from fundingStream.id
    -- Split by '::' and take the 2nd element
    CASE 
        WHEN list_contains(
            list_transform(fundings, f -> f.shortName), 
            'EC'
        ) THEN (
            SELECT string_split(
                list_filter(
                    list_transform(fundings, f -> f.fundingStream.id),
                    x -> x LIKE 'EC::%'
                )[1],  -- first EC funding stream
                '::'
            )[2]  -- second segment = framework
        )
        ELSE NULL
    END as eu_framework,
    
    -- Keep full fundings
    fundings,
    
    -- other columns...
FROM project
"""
q = "select * from projects_clean where eu_framework is not null limit 10"
# display(con.execute(q).df())

In [16]:
### For each PID scheme (ROR, GRID etc.), how many projects connect to orgs with that scheme?
#
# 2658616 without scheme

### With scheme
q = """ 
WITH project_schemes AS (
    SELECT DISTINCT 
        p.id AS project_id,
        op.scheme
    FROM project p
    JOIN relation r 
        ON r.source = p.id 
        AND r.sourceType = 'project'
        AND r.relType.type = 'participation'
    JOIN organization o 
        ON r.target = o.id 
        AND r.targetType = 'organization'
    JOIN organization_pids op 
        ON op.org_id = o.id
)
SELECT 
    scheme,
    COUNT(DISTINCT project_id) AS projects_count
FROM project_schemes
GROUP BY scheme
ORDER BY projects_count DESC;
"""
display(con.execute(q).df())

### Schemeless
q = """ 
WITH project_schemes AS (
    SELECT DISTINCT p.id AS project_id
    FROM project p
    JOIN relation r 
        ON r.source = p.id 
        AND r.sourceType = 'project'
        AND r.relType.type = 'participation'
    JOIN organization o 
        ON r.target = o.id 
        AND r.targetType = 'organization'
    JOIN organization_pids op 
        ON op.org_id = o.id
)
SELECT COUNT(*) AS schemeless_projects
FROM project p
WHERE p.id NOT IN (SELECT project_id FROM project_schemes);
"""
display(con.execute(q).df())

CatalogException: Catalog Error: Table with name organization_pids does not exist!
Did you mean "organization"?

LINE 14:     JOIN organization_pids op 
                  ^

In [17]:
### What types of project→org relationships exist?
# Result: No distinction between coordinator / participant

q = """ 
SELECT 
    r.relType.type,
    r.relType.name,
    COUNT(*) as count,
    COUNT(DISTINCT r.source) as unique_projects,
    COUNT(DISTINCT r.target) as unique_orgs
FROM relation r
WHERE r.sourceType = 'project' 
  AND r.targetType = 'organization'
GROUP BY r.relType.type, r.relType.name
ORDER BY count DESC;
"""
display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,type,name,count,unique_projects,unique_orgs
0,participation,hasParticipant,4568712,3372450,309580


In [19]:
### Date range sanity check: low and high years in project and publication dates
# We need to sanitize: year < 1000 AND or also > 3000

q = """                                                                                                                                        
SELECT 'project.startDate' AS col, MIN(year(startDate)) AS min_year, MAX(year(startDate)) AS max_year,                                         
    COUNT(*) FILTER (WHERE year(startDate) < 1000) AS below_1000,                                                                              
    COUNT(*) FILTER (WHERE year(startDate) > 3000) AS above_3000                                                                               
FROM project WHERE startDate IS NOT NULL                                                                                                       
UNION ALL                                                                                                                                      
SELECT 'project.endDate', MIN(year(endDate)), MAX(year(endDate)),                                                                              
    COUNT(*) FILTER (WHERE year(endDate) < 1000),                                                                                              
    COUNT(*) FILTER (WHERE year(endDate) > 3000)                                                                                               
FROM project WHERE endDate IS NOT NULL                                                                                                         
UNION ALL                                                                                                                                      
SELECT 'work.publicationDate', MIN(year(publicationDate)), MAX(year(publicationDate)),
    COUNT(*) FILTER (WHERE year(publicationDate) < 1000),                                                                                      
    COUNT(*) FILTER (WHERE year(publicationDate) > 3000)                                                                                       
FROM work WHERE publicationDate IS NOT NULL                                                                                           
"""
display(con.execute(q).df())  

,col,min_year,max_year,below_1000,above_3000
0,project.startDate,7,2125,18,0
1,project.endDate,1900,9999,0,3837
2,work.publicationDate,1,9999,14178,7198


## Entity Publication

In [14]:
### Sample Rows

q = """ 
SELECT * FROM work
LIMIT 30
OFFSET 200000001
"""
display(con.execute(q).df())

,id,openaireId,title,publicationDate,publisher,openAccessColor,isGreen,isInDiamondJournal,publiclyFunded,language,bestAccessRight,authors,subjects,descriptions,pids,sources,formats,instances,citationCount,influence,views,container
0,9415925887746482857,doi_________::cf42da24575e3e7a875f09b3c933cad2,Proceedings of the First Annual Conference of ...,1971-04-01,SAGE Publications,None,False,False,False,"{'code': 'eng', 'label': 'English'}","{'code': 'c_14cb', 'label': 'CLOSED', 'scheme'...",<NA>,<NA>,<NA>,"[{'scheme': 'doi', 'value': '10.1177/004728757...",[Crossref],<NA>,[{'license': 'https://journals.sagepub.com/pag...,0.0,2.583162e-09,<NA>,"{'name': 'Travel Research Bulletin', 'issnPrin..."
1,6460611479800920237,doi_________::cf5356eb85fa4ae2b5d201947ee0054f,EMS Insider,2011-09-01,Elsevier BV,None,False,False,False,"{'code': 'eng', 'label': 'English'}","{'code': 'c_14cb', 'label': 'CLOSED', 'scheme'...",<NA>,<NA>,<NA>,"[{'scheme': 'doi', 'value': '10.1016/s1081-450...",[Crossref],<NA>,"[{'license': 'Elsevier TDM', 'accessRight': {'...",0.0,2.583162e-09,<NA>,"{'name': 'EMS Insider', 'issnPrinted': '1081-4..."
2,2280388855922583727,doi_________::cf5feed634d2f31b980fc774f9bddfdb,Note,2009-03-01,Franco Angeli,None,False,False,False,"{'code': 'und', 'label': 'Undetermined'}",<NA>,"[{'fullName': 'Antonio Maturo', 'name': 'Anton...","[{'subject': {'scheme': 'FOS', 'value': '03 me...",<NA>,"[{'scheme': 'doi', 'value': '10.3280/ses2009-0...",[Crossref],<NA>,"[{'license': None, 'accessRight': None, 'type'...",0.0,2.583162e-09,<NA>,"{'name': 'SALUTE E SOCIETÀ', 'issnPrinted': '1..."
3,5181840263665961292,doi_________::cf61e7de02170b73d3f41f6e283c3a6f,Images et sons,2004-08-01,CAIRN,None,False,False,False,"{'code': 'und', 'label': 'Undetermined'}",<NA>,<NA>,<NA>,<NA>,"[{'scheme': 'doi', 'value': '10.3917/ving.083....",[Crossref],<NA>,"[{'license': None, 'accessRight': None, 'type'...",0.0,2.583162e-09,<NA>,"{'name': 'Vingtième Siècle. Revue d'histoire',..."
4,14699259981232195458,doi_________::cf69e1c8daf2789f79c93e0b1a08842e,Решение задачи построения математической модел...,2021-10-04,NPG Publishing,None,False,False,False,"{'code': 'und', 'label': 'Undetermined'}","{'code': 'c_abf2', 'label': 'OPEN', 'scheme': ...",<NA>,<NA>,[<jats:p>Актуальность выбранной темы обусловле...,"[{'scheme': 'doi', 'value': '10.24108/preprint...",[Crossref],<NA>,"[{'license': None, 'accessRight': None, 'type'...",0.0,2.583162e-09,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,17176906031274916464,doi_________::d0ec79280e351fa8b4a57b153177def6,Building modernization located in the conserva...,2022-05-16,Centrum Rzeczoznawstwa Budowlanego Sp z. o.o.,gold,False,False,False,"{'code': 'und', 'label': 'Undetermined'}","{'code': 'c_abf2', 'label': 'OPEN', 'scheme': ...","[{'fullName': 'Maciej NIEDOSTATKIEWICZ', 'name...","[{'subject': {'scheme': 'SDG', 'value': '11. S...",[<jats:p>The paper presents a description of t...,"[{'scheme': 'doi', 'value': '10.37105/iboa.133'}]",[Crossref],<NA>,"[{'license': 'CC BY SA', 'accessRight': {'code...",0.0,2.583162e-09,<NA>,{'name': 'Inżynieria Bezpieczeństwa Obiektów A...
26,16523867716069600531,doi_________::d1336dd444362f18103bcc9361205204,Bücherschau,1937-05-01,Walter de Gruyter GmbH,None,False,False,False,"{'code': 'eng', 'label': 'English'}",<NA>,<NA>,<NA>,<NA>,"[{'scheme': 'doi', 'value': '10.1515/zpch-1937...",[Crossref],<NA>,"[{'license': None, 'accessRight': None, 'type'...",0.0,2.583162e-09,<NA>,{'name': 'Zeitschrift für Physikalische Chemie...
27,5853123668159886629,doi_________::d16432067ddfb2ff0744dfaf85741f13,Conclusion,2018-09-08,Presses de l'Université du Québec,None,False,False,False,"{'code': 'und', 'label': 'Undetermined'}",<NA>,<NA>,<NA>,<NA>,"[{'scheme': 'doi', 'value': '10.2307/j.ctv5j01...","[Crossref, Fondements et pratiques de l'éducat...",<NA>,"[{'license': None, 'accessRight': None, 'type'...",0.0,2.583162e-09,<NA>,<NA>
28,6024906342842980676,doi

In [ ]:
### Descriptions distribution

q = """
SELECT 
    CASE 
        WHEN descriptions IS NULL THEN -1
        ELSE len(descriptions)
    END AS num_descriptions,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM work
GROUP BY num_descriptions
ORDER BY num_descriptions
"""
display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

KeyboardInterrupt: 

In [ ]:
### Check null count for each Column

q = """
SELECT 
    column_name,
    null_count,
    total_rows,
    ROUND(100.0 * null_count / total_rows, 2) as pct_null
FROM (
    SELECT 
        'id' as column_name,
        COUNT(*) FILTER (WHERE id IS NULL) as null_count,
        COUNT(*) as total_rows
    FROM work
    
    UNION ALL
    
    SELECT 'type', COUNT(*) FILTER (WHERE type IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'mainTitle', COUNT(*) FILTER (WHERE mainTitle IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'subTitle', COUNT(*) FILTER (WHERE subTitle IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'publicationDate', COUNT(*) FILTER (WHERE publicationDate IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'publisher', COUNT(*) FILTER (WHERE publisher IS NULL), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'embargoEndDate', COUNT(*) FILTER (WHERE embargoEndDate IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'dateOfCollection', COUNT(*) FILTER (WHERE dateOfCollection IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'lastUpdateTimeStamp', COUNT(*) FILTER (WHERE lastUpdateTimeStamp IS NULL), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'openAccessColor', COUNT(*) FILTER (WHERE openAccessColor IS NULL), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'isGreen', COUNT(*) FILTER (WHERE isGreen IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'isInDiamondJournal', COUNT(*) FILTER (WHERE isInDiamondJournal IS NULL), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'publiclyFunded', COUNT(*) FILTER (WHERE publiclyFunded IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'language', COUNT(*) FILTER (WHERE language IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'bestAccessRight', COUNT(*) FILTER (WHERE bestAccessRight IS NULL), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'authors', COUNT(*) FILTER (WHERE authors IS NULL OR len(authors) = 0), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'countries', COUNT(*) FILTER (WHERE countries IS NULL OR len(countries) = 0), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'subjects', COUNT(*) FILTER (WHERE subjects IS NULL OR len(subjects) = 0), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'descriptions', COUNT(*) FILTER (WHERE descriptions IS NULL OR len(descriptions) = 0), COUNT(*) FROM work
    UNION ALL
    SELECT 'originalIds', COUNT(*) FILTER (WHERE originalIds IS NULL OR len(originalIds) = 0), COUNT(*) FROM work
    UNION ALL
    SELECT 'pids', COUNT(*) FILTER (WHERE pids IS NULL OR len(pids) = 0), COUNT(*) FROM work
    UNION ALL
    SELECT 'sources', COUNT(*) FILTER (WHERE sources IS NULL OR len(sources) = 0), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'formats', COUNT(*) FILTER (WHERE formats IS NULL OR len(formats) = 0), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'contributors', COUNT(*) FILTER (WHERE contributors IS NULL OR len(contributors) = 0), COUNT(*) FROM work
    UNION ALL
    SELECT 'coverages', COUNT(*) FILTER (WHERE coverages IS NULL), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'container', COUNT(*) FILTER (WHERE container IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'instances', COUNT(*) FILTER (WHERE instances IS NULL OR len(instances) = 0), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'indicators', COUNT(*) FILTER (WHERE indicators IS NULL), COUNT(*) FROM work
    UNION ALL
    SELECT 'documentationUrls', COUNT(*) FILTER (WHERE documentationUrls IS NULL), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'codeRepositoryUrl', COUNT(*) FILTER (WHERE codeRepositoryUrl IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'programmingLanguage', COUNT(*) FILTER (WHERE programmingLanguage IS NULL), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'contactPeople', COUNT(*) FILTER (WHERE contactPeople IS NULL), COUNT(*) FROM work
    UNION ALL
    -- SELECT 'contactGroups', COUNT(*) FILTER (WHERE contactGroups IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'tools', COUNT(*) FILTER (WHERE tools IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'size', COUNT(*) FILTER (WHERE size IS NULL), COUNT(*) FROM work
    -- UNION ALL
    -- SELECT 'version', COUNT(*) FILTER (WHERE version IS NULL), COUNT(*) FROM work
    -- UNION ALL
    SELECT 'geoLocations', COUNT(*) FILTER (WHERE geoLocations IS NULL), COUNT(*) FROM work
)
ORDER BY pct_null DESC
"""

display(con.execute(q).df())

## Entity relation

* Do i need to precompute Works and Projects impact, what is already there
* Remove all relations.type? == product where is not project??????

In [15]:
q = """ 
SELECT * FROM relation LIMIT 3
"""
display(con.execute(q).df())

,source,sourceType,target,targetType,relType,provenance,validated
0,7394781489291612645,product,18150248833008221797,organization,"{'name': 'hasAuthorInstitution', 'type': 'affi...","{'provenance': 'Inferred by OpenAIRE', 'trust'...",False
1,4611416004720601728,product,18150248833008221797,organization,"{'name': 'hasAuthorInstitution', 'type': 'affi...","{'provenance': 'Inferred by OpenAIRE', 'trust'...",False
2,1943491943382936829,product,18150248833008221797,organization,"{'name': 'hasAuthorInstitution', 'type': 'affi...","{'provenance': 'Inferred by OpenAIRE', 'trust'...",False


In [3]:
### Check all unique sourceType and targetType values

q = """
SELECT 
    sourceType,
    targetType,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct
FROM relation
GROUP BY sourceType, targetType
ORDER BY count DESC
"""
display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sourceType,targetType,count,pct
0,product,organization,268313783,47.73
1,organization,product,268313783,47.73
2,project,product,8220760,1.46
3,product,project,8220760,1.46
4,project,organization,4568712,0.81
5,organization,project,4568712,0.81


In [4]:
### Check ID prefixes of products in relations

q = """
SELECT 
    SUBSTRING(source, 1, 12) as id_prefix,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct
FROM relation
WHERE sourceType = 'product'
GROUP BY id_prefix
ORDER BY count DESC
LIMIT 20
"""
display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id_prefix,count,pct
0,doi_dedup___,223717794,80.90
1,doi_________,22320185,8.07
2,dedup_wf_002,13247272,4.79
3,pmid________,2155889,0.78
4,pmid_dedup__,1136784,0.41
5,od_______177,580252,0.21
6,a9ac50f576aa,474983,0.17
7,od_____10049,334374,0.12
8,r3c4b2081b22,259387,0.09
9,od______3686,222980,0.08


## Impact Metrics

In [ ]:
### Question: What impact metrics exist for publications?
# Result:
# total_publications	has_indicators	has_citationImpact	has_usageCounts	avg_citations	avg_downloads	avg_views
# 205841448	            200351379	    200347669	        3600953	        10.627821	    57.977084	    35.391892
# Results without null:
# total_publications	has_indicators	has_citationImpact	has_usageCounts	avg_citations	avg_downloads	avg_views
# 205841448	            200351379	    200351379	        200351379	    10.627821	    57.977084	    35.391892


q = """
SELECT
    COUNT(*) AS total_publications,
    COUNT(indicators) AS has_indicators,
    COUNT(indicators->'citationImpact') AS has_citationImpact,
    COUNT(indicators->'usageCounts') AS has_usageCounts,
    AVG(CAST(NULLIF(indicators->'citationImpact'->>'citationCount','') AS INTEGER)) AS avg_citations,
    AVG(CAST(NULLIF(indicators->'usageCounts'->>'downloads','') AS INTEGER)) AS avg_downloads,
    AVG(CAST(NULLIF(indicators->'usageCounts'->>'views','') AS INTEGER)) AS avg_views
FROM work
"""

display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_publications,has_indicators,has_citationImpact,has_usageCounts,avg_citations,avg_downloads,avg_views
0,205841448,200351379,200351379,200351379,10.627821,57.977084,35.391892


In [ ]:
### Top 3 cited papers

q = """
SELECT
    id,
    mainTitle,
    CAST(NULLIF(indicators->'citationImpact'->>'citationCount','') AS INTEGER) AS citations
FROM work
WHERE indicators->'citationImpact'->>'citationCount' IS NOT NULL
ORDER BY citations DESC
LIMIT 3
"""

display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,mainTitle,citations
0,doi_dedup___::6013564d557628905abf3c0f70e51675,PROTEIN MEASUREMENT WITH THE FOLIN PHENOL REAGENT,340187
1,doi_dedup___::ee253ec27e28ed6b0df7da99c853917f,Cleavage of Structural Proteins during the Ass...,262820
2,doi_dedup___::ec319731a12c0b83e5d4bf781693cdac,A Rapid and Sensitive Method for the Quantitat...,248877


In [8]:
### Check indicators struct: column paths for citationCount, influence, views
# Confirms staging.py extraction: indicators.citationImpact.citationCount / .influence / indicators.usageCounts.views

q = """                                                                                                                                        
SELECT                                                                                                                                         
    COUNT(*)                                            AS total,                                                                              
    COUNT(indicators)                                   AS has_indicators,                                                                     
    COUNT(indicators.citationImpact.citationCount)      AS has_citationCount,                                                                  
    COUNT(indicators.citationImpact.influence)          AS has_influence,                                                                      
    COUNT(indicators.usageCounts.views)                 AS has_views,                                                                          
    AVG(indicators.citationImpact.citationCount)        AS avg_citationCount,                                                                  
    AVG(indicators.citationImpact.influence)            AS avg_influence,                                                                      
    AVG(indicators.usageCounts.views)                   AS avg_views                                                                           
FROM work                                                                                                                               
"""                                                                                                                                            
display(con.execute(q).df())    

,total,has_indicators,has_citationCount,has_influence,has_views,avg_citationCount,avg_influence,avg_views
0,205841448,200351379,200347669,200347669,3600953,10.627821,3.496346e-09,35.391892


# Connecting to Cordis

In [9]:
# Check Project ID prefixes
q_proj_ids = """
SELECT 
    split_part(id, '::', 1) as prefix, 
    count(*) as cnt
FROM project
GROUP BY 1
ORDER BY cnt DESC;
"""
display(con.execute(q_proj_ids).df())

# Check Organization ID prefixes
q_org_ids = """
SELECT 
    split_part(id, '::', 1) as prefix, 
    count(*) as cnt
FROM organization
GROUP BY 1
ORDER BY cnt DESC;
"""
display(con.execute(q_org_ids).df())

,prefix,cnt
0,nih_________,2277712
1,nsf_________,592489
2,ukri________,175092
3,fct_________,94440
4,snsf________,90082
...,...,...
264,501100010808,1
265,501100000274,1
266,nserc_______,1
267,501100001589,1


,prefix,cnt
0,pending_org_,345425
1,openorgs____,102726
2,edtech______,4
3,ror_________,3
4,ukri________,2
5,fairsharing_,1


In [10]:
### Count all EC Projects: 115447
# From 155 979 (with archived content) officially on cordis

q = """ 
SELECT 
    count(*)
FROM project
WHERE list_transform(fundings, x -> x.shortName).list_contains('EC');
"""
display(con.execute(q).df())

,count_star()
0,115447


In [ ]:
### Breakdown of EC Projects by Programme
q = """
SELECT 
    f.fundingStream, 
    count(*) as project_count
FROM project, 
     UNNEST(fundings) as f
WHERE f.shortName = 'EC'
GROUP BY 1
ORDER BY project_count DESC;
"""
display(con.execute(q).df())